In [1]:
import math
import copy
from dataclasses import dataclass

import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Function


In [2]:
@dataclass
class Config:
    batch_size: int = 256
    num_workers: int = 8
    pin_memory: bool = True
    persistent_workers: bool = True
    prefetch_factor: int = 4
    lr: float = 1e-3
    weight_decay: float = 1e-4
    num_epochs: int = 100
    val_split: int = 500
    train_split: int = 50000 - val_split
    use_progressive_ede: bool = False  # AdaBin script supports progressive k/t as optional
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


cfg = Config()
print(cfg.device)


cuda


In [3]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4, padding_mode="reflect"),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
        transforms.RandomErasing(
            p=0.25,
            scale=(0.02, 0.2),
            ratio=(0.3, 3.3),
            value="random",
        ),
    ]
)

test_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)


In [4]:
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform,
)

train_set, val_set = torch.utils.data.random_split(
    train_dataset,
    [cfg.train_split, cfg.val_split],
)

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
    prefetch_factor=cfg.prefetch_factor if cfg.num_workers > 0 else None,
)

# Preserves your original validation split behavior.
# Note: val_set inherits train_transform because it is split from train_dataset.
val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
    prefetch_factor=cfg.prefetch_factor if cfg.num_workers > 0 else None,
)

test_set = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform,
)

test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
    prefetch_factor=cfg.prefetch_factor if cfg.num_workers > 0 else None,
)


Files already downloaded and verified
Files already downloaded and verified


In [5]:
class BinaryQuantize(Function):
    """Binary quantize function used by AdaBin code, inherited from IR-Net."""

    @staticmethod
    def forward(ctx, input, k, t):
        ctx.save_for_backward(input, k, t)
        return torch.sign(input)

    @staticmethod
    def backward(ctx, grad_output):
        input, k, t = ctx.saved_tensors
        grad_input = k * t * (1.0 - torch.tanh(input * t).pow(2)) * grad_output
        return grad_input, None, None


In [6]:
class Maxout(nn.Module):
    """AdaBin nonlinear function."""

    def __init__(self, channel, neg_init=0.25, pos_init=1.0, spatial=True):
        super().__init__()
        self.neg_scale = nn.Parameter(neg_init * torch.ones(channel))
        self.pos_scale = nn.Parameter(pos_init * torch.ones(channel))
        self.spatial = spatial
        self.relu = nn.ReLU()

    def forward(self, x):
        if self.spatial:
            shape = (1, -1, 1, 1)
        else:
            shape = (1, -1)

        pos = self.pos_scale.view(*shape) * self.relu(x)
        neg = self.neg_scale.view(*shape) * self.relu(-x)
        return pos - neg


In [7]:
class BinaryActivation(nn.Module):
    """AdaBin learnable activation center beta_a and distance alpha_a."""

    def __init__(self):
        super().__init__()
        self.alpha_a = nn.Parameter(torch.tensor(1.0))
        self.beta_a = nn.Parameter(torch.tensor(0.0))

    def gradient_approx(self, x):
        out_forward = torch.sign(x)

        mask1 = x < -1
        mask2 = x < 0
        mask3 = x < 1

        mask1 = mask1.to(dtype=x.dtype)
        mask2 = mask2.to(dtype=x.dtype)
        mask3 = mask3.to(dtype=x.dtype)

        out1 = (-1.0) * mask1 + (x * x + 2.0 * x) * (1.0 - mask1)
        out2 = out1 * mask2 + (-x * x + 2.0 * x) * (1.0 - mask2)
        out3 = out2 * mask3 + 1.0 * (1.0 - mask3)

        return out_forward.detach() - out3.detach() + out3

    def forward(self, x):
        alpha = self.alpha_a.clamp_min(1e-6)
        x = (x - self.beta_a) / alpha
        x = self.gradient_approx(x)
        return alpha * (x + self.beta_a)


In [8]:
class AdaBin_Conv2d(nn.Conv2d):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        padding=0,
        dilation=1,
        groups=1,
        bias=False,
        a_bit=1,
        w_bit=1,
        eps=1e-6,
    ):
        super().__init__(
            in_channels,
            out_channels,
            kernel_size,
            stride,
            padding,
            dilation,
            groups,
            bias,
        )

        self.a_bit = a_bit
        self.w_bit = w_bit
        self.eps = eps

        self.register_buffer("k", torch.tensor([10.0]))
        self.register_buffer("t", torch.tensor([0.1]))

        self.binary_a = BinaryActivation()

        kh, kw = self.kernel_size
        self.filter_size = kh * kw * (self.in_channels // self.groups)

        self.perturb_prob = 0.0
        self.perturb_seed = 1234

    def _maybe_flip_binary_weights(self, wb):
        if self.perturb_prob <= 0.0:
            return wb

        gen = torch.Generator(device=wb.device)
        gen.manual_seed(int(self.perturb_seed))
        mask = torch.rand(wb.shape, generator=gen, device=wb.device) < self.perturb_prob
        return torch.where(mask, -wb, wb)

    def forward(self, inputs):
        if self.a_bit == 1:
            inputs = self.binary_a(inputs)

        if self.w_bit == 1:
            w = self.weight

            beta_w = w.mean(dim=(1, 2, 3), keepdim=True)
            alpha_w = torch.sqrt(((w - beta_w) ** 2).sum(dim=(1, 2, 3), keepdim=True) / self.filter_size)
            alpha_w = alpha_w.clamp_min(self.eps)

            w_norm = (w - beta_w) / alpha_w
            wb = BinaryQuantize.apply(w_norm, self.k, self.t)
            wb = self._maybe_flip_binary_weights(wb)

            weight = wb * alpha_w + beta_w
        else:
            weight = self.weight

        return F.conv2d(
            inputs,
            weight,
            self.bias,
            self.stride,
            self.padding,
            self.dilation,
            self.groups,
        )


In [9]:
class AdaBin_Linear(nn.Linear):
    def __init__(
        self,
        in_features,
        out_features,
        bias=True,
        a_bit=1,
        w_bit=1,
        eps=1e-6,
    ):
        super().__init__(in_features, out_features, bias=bias)

        self.a_bit = a_bit
        self.w_bit = w_bit
        self.eps = eps
        self.filter_size = in_features

        self.register_buffer("k", torch.tensor([10.0]))
        self.register_buffer("t", torch.tensor([0.1]))

        self.binary_a = BinaryActivation()

        self.perturb_prob = 0.0
        self.perturb_seed = 4321

    def _maybe_flip_binary_weights(self, wb):
        if self.perturb_prob <= 0.0:
            return wb

        gen = torch.Generator(device=wb.device)
        gen.manual_seed(int(self.perturb_seed))
        mask = torch.rand(wb.shape, generator=gen, device=wb.device) < self.perturb_prob
        return torch.where(mask, -wb, wb)

    def forward(self, inputs):
        if self.a_bit == 1:
            inputs = self.binary_a(inputs)

        if self.w_bit == 1:
            w = self.weight

            beta_w = w.mean(dim=1, keepdim=True)
            alpha_w = torch.sqrt(((w - beta_w) ** 2).sum(dim=1, keepdim=True) / self.filter_size)
            alpha_w = alpha_w.clamp_min(self.eps)

            w_norm = (w - beta_w) / alpha_w
            wb = BinaryQuantize.apply(w_norm, self.k, self.t)
            wb = self._maybe_flip_binary_weights(wb)

            weight = wb * alpha_w + beta_w
        else:
            weight = self.weight

        return F.linear(inputs, weight, self.bias)


In [10]:
def update_adabin_ede_params(model, epoch, total_epochs, t_min=0.1, t_max=10.0):
    """Optional progressive k/t schedule, matching the AdaBin script's progressive mode."""

    progress = epoch / max(total_epochs - 1, 1)
    t = t_min * (10 ** (progress * math.log10(t_max / t_min)))
    k = max(1.0 / t, 1.0)

    for module in model.modules():
        if isinstance(module, (AdaBin_Conv2d, AdaBin_Linear)):
            module.k.fill_(k)
            module.t.fill_(t)

    return k, t


In [11]:
class Network(nn.Module):
    def __init__(
        self,
        input_channels=3,
        num_classes=10,
        image_size=32,
        dropout=0.25,
        n_ratio=1.0,
        use_maxout=False,
    ):
        super().__init__()

        self.img_size = image_size
        self.cin = input_channels
        self.cout1 = 32
        self.cout2 = 32

        self.fcin = self.cout2 * (16 ** 2)
        self.fcout1 = 128
        self.fcout2 = 128
        self.fcout3 = num_classes

        self.n_ratio = n_ratio

        self.block1 = nn.Sequential(
            nn.Conv2d(self.cin, self.cout1, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(self.cout1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
        )

        block2_activation = Maxout(self.cout2, spatial=True) if use_maxout else nn.ReLU()
        block3_activation = Maxout(self.fcout1, spatial=False) if use_maxout else nn.ReLU()
        block4_activation = Maxout(self.fcout2, spatial=False) if use_maxout else nn.ReLU()

        self.block2 = nn.Sequential(
            AdaBin_Conv2d(self.cout1, self.cout2, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(self.cout2),
            block2_activation,
        )

        self.block3 = nn.Sequential(
            AdaBin_Linear(self.fcin, self.fcout1),
            nn.BatchNorm1d(self.fcout1),
            block3_activation,
            nn.Dropout(dropout),
        )

        self.block4 = nn.Sequential(
            AdaBin_Linear(self.fcout1, self.fcout2),
            nn.BatchNorm1d(self.fcout2),
            block4_activation,
            nn.Dropout(dropout),
        )

        self.head = nn.Linear(self.fcout2, self.fcout3)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)

        x = torch.flatten(x, start_dim=1)

        x = self.block3(x)
        x = self.block4(x)
        x = self.head(x)

        return x


In [12]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch, labels in loader:
        batch = batch.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(batch)
        loss = criterion(outputs, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += batch_size

    return total_loss / total, correct / total


In [13]:
model = Network(n_ratio=5.0, use_maxout=False).to(cfg.device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=10,
)

best_val_acc = 0.0
best_state_dict = None


In [14]:
for epoch in range(cfg.num_epochs):
    if cfg.use_progressive_ede:
        k, t = update_adabin_ede_params(model, epoch, cfg.num_epochs)
    else:
        k, t = None, None

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch, labels in train_loader:
        batch = batch.to(cfg.device, non_blocking=True)
        labels = labels.to(cfg.device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(batch)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += batch_size

    train_loss = running_loss / total
    train_acc = correct / total

    val_loss, val_acc = evaluate(model, val_loader, criterion, cfg.device)
    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model.state_dict())

    kt_msg = f"k={k:.4f} t={t:.4f} " if cfg.use_progressive_ede else ""

    print(
        f"Epoch [{epoch + 1}/{cfg.num_epochs}] "
        f"{kt_msg}"
        f"Train loss: {train_loss:.4f} "
        f"Train acc: {train_acc:.4f} "
        f"Val loss: {val_loss:.4f} "
        f"Val acc: {val_acc:.4f}"
    )

print(f"Best Validation Accuracy: {best_val_acc:.4f}")


Epoch [1/100] Train loss: 2.1104 Train acc: 0.2206 Val loss: 2.0678 Val acc: 0.2500
Epoch [2/100] Train loss: 2.0156 Train acc: 0.2656 Val loss: 1.9852 Val acc: 0.2860
Epoch [3/100] Train loss: 1.9655 Train acc: 0.2862 Val loss: 1.9079 Val acc: 0.2960
Epoch [4/100] Train loss: 1.9233 Train acc: 0.2995 Val loss: 1.8365 Val acc: 0.3340
Epoch [5/100] Train loss: 1.8934 Train acc: 0.3158 Val loss: 1.7900 Val acc: 0.3380
Epoch [6/100] Train loss: 1.8675 Train acc: 0.3230 Val loss: 1.7757 Val acc: 0.3680
Epoch [7/100] Train loss: 1.8525 Train acc: 0.3296 Val loss: 1.7716 Val acc: 0.3520
Epoch [8/100] Train loss: 1.8340 Train acc: 0.3408 Val loss: 1.6663 Val acc: 0.4140
Epoch [9/100] Train loss: 1.8195 Train acc: 0.3420 Val loss: 1.7439 Val acc: 0.3600
Epoch [10/100] Train loss: 1.8065 Train acc: 0.3495 Val loss: 1.7125 Val acc: 0.3860
Epoch [11/100] Train loss: 1.7926 Train acc: 0.3538 Val loss: 1.6253 Val acc: 0.4220
Epoch [12/100] Train loss: 1.7855 Train acc: 0.3577 Val loss: 1.6904 Val a

In [15]:
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

clean_state_dict = copy.deepcopy(model.state_dict())

clean_test_loss, clean_test_acc = evaluate(model, test_loader, criterion, cfg.device)
print(
    f"Clean Test Loss: {clean_test_loss:.4f} "
    f"Clean Test Acc: {clean_test_acc:.4f}"
)


Clean Test Loss: 0.8895 Clean Test Acc: 0.6830


In [16]:
def set_adabin_bit_flip_prob(model, p, base_seed=1234):
    layer_idx = 0

    for layer in model.modules():
        if isinstance(layer, (AdaBin_Conv2d, AdaBin_Linear)):
            layer.perturb_prob = float(p)
            layer.perturb_seed = int(base_seed + layer_idx)
            layer_idx += 1


def reset_adabin_bit_flips(model):
    for layer in model.modules():
        if isinstance(layer, (AdaBin_Conv2d, AdaBin_Linear)):
            layer.perturb_prob = 0.0


In [17]:
device = next(model.parameters()).device
flip_percentages = list(range(0, 51, 5))

results = []

for pct in flip_percentages:
    p = pct / 100.0

    set_adabin_bit_flip_prob(
        model=model,
        p=p,
        base_seed=1234,
    )

    test_loss, test_acc = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
    )

    results.append(
        {
            "flip_percent": pct,
            "flip_prob": p,
            "test_loss": test_loss,
            "test_acc": test_acc,
            "test_acc_percent": 100 * test_acc,
        }
    )

    print(
        f"Flip {pct:2d}% | "
        f"Test loss: {test_loss:.4f} | "
        f"Test acc: {100 * test_acc:.2f}%"
    )

reset_adabin_bit_flips(model)


Flip  0% | Test loss: 0.8895 | Test acc: 68.30%
Flip  5% | Test loss: 1.0660 | Test acc: 62.29%
Flip 10% | Test loss: 1.4523 | Test acc: 49.46%
Flip 15% | Test loss: 1.7691 | Test acc: 40.14%
Flip 20% | Test loss: 2.1858 | Test acc: 21.97%
Flip 25% | Test loss: 2.5193 | Test acc: 11.30%
Flip 30% | Test loss: 2.6666 | Test acc: 10.32%
Flip 35% | Test loss: 2.8150 | Test acc: 10.13%
Flip 40% | Test loss: 2.7673 | Test acc: 9.95%
Flip 45% | Test loss: 2.6594 | Test acc: 10.14%
Flip 50% | Test loss: 2.5962 | Test acc: 9.66%
